[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adkipnis/metabeta/blob/main/demos/03-priors.ipynb)

# priors and model comparison

metabeta takes the prior specification as an input, and permits model comparison via Bayes
factors.<br>
For demonstration purposes, this notebook fits simulated data under known ground truth parameters.

In [ ]:
%pip install -q metabeta==0.5.1

## Load a checkpoint

In [1]:
import numpy as np
import pandas as pd
import torch

from metabeta import Api

device = 'cuda' if torch.cuda.is_available() else 'cpu'
mb = Api.from_pretrained('gaussian', device=device)  # the test score is continuous

Resolving metabeta 'normal' checkpoint from Hugging Face (adkipnis/metabeta@v1)...


Using checkpoint: /Users/alex/.cache/huggingface/hub/models--adkipnis--metabeta/snapshots/66c398ef31d65400f9ce6cff262265a9cd895a4d/metabeta-normal.pt


Loading metabeta checkpoint on cpu...


Metabeta checkpoint ready.


## Simulate the data

Imagine that students in 15 schools take a test:

- **Outcome**: the test `score`.
- **Predictors**: weekly study `hours` (which raise the score) and the `commute` in minutes
  (which doesn't).
- **Groups**: each school gets its own intercept and slope.

In [2]:
rng = np.random.default_rng(1)
n_schools = 15
school = np.repeat(np.arange(n_schools), rng.integers(6, 13, size=n_schools))  # (n,) 6–12 each
hours = rng.normal(5, 2, size=school.size).clip(0)
commute = rng.normal(30, 10, size=school.size).clip(5)

truth = pd.Series(
    {
        'Intercept': 60.0,
        'hours': 2.0,
        'commute': 0.0,
        'σ_Intercept': 6.0,
        'σ_hours': 1.0,
        'Corr(Intercept, hours)': -0.3,
        'σ_Residual': 8.0,
    }
)
sd_school = truth[['σ_Intercept', 'σ_hours']].to_numpy()
corr = np.array([[1, truth['Corr(Intercept, hours)']], [truth['Corr(Intercept, hours)'], 1]])
u = rng.multivariate_normal(np.zeros(2), corr * np.outer(sd_school, sd_school), size=n_schools)
score = (
    truth['Intercept']
    + u[school, 0]
    + (truth['hours'] + u[school, 1]) * hours
    + truth['commute'] * commute
    + rng.normal(0, truth['σ_Residual'], size=school.size)
)
df = pd.DataFrame(
    {'score': score, 'hours': hours, 'commute': commute, 'school': [f's{g:02d}' for g in school]}
)

sizes = df.groupby('school').size()
print(f'{len(df)} students in {len(sizes)} schools with {sizes.min()}–{sizes.max()} students each')

137 students in 15 schools with 6–12 students each


## Specifying priors

`sample(priors=...)` takes a dictionary with up to four keys.<br>
Terms are named as in the formula.<br>
Anything unspecified keeps its default.

| key | parameters | per term | supported families | default |
|---|---|---|---|---|
| `fixed` | intercept and slopes | `{'nu': mean, 'tau': scale}` | `'normal'`, `'student'` | N(0, 2.5) |
| `random_sd` | standard deviation (SD) of each random effect | `{'tau': scale}` | `'halfnormal'`, `'halfstudent'`, `'exponential'` | HN(2.5) |
| `sigma_eps` | residual SD | `{'tau': scale}`, without terms | as `random_sd` | HT₅(1) |
| `corr_rfx` | correlations of the random effects | `{'eta': concentration}`, without terms | LKJ | LKJ(1) |

- A `'family'` entry sets the family of all the key's terms.
- N and HN are the normal and half-normal.
- t₅ and HT₅ are the Student-t and half-Student-t with five degrees of freedom.
- An exponential prior's `tau` is its mean.
- An LKJ (Lewandowski–Kurowicka–Joe) concentration above 1 favors weak correlations.
- `sigma_eps` exists for the Gaussian family only, `corr_rfx` from two random effects onwards.

### The standardized scale

The priors apply on the scale the networks were trained on:

- `sample` standardizes each numeric predictor and, for the Gaussian family, the response to mean
  0 and SD 1.
- The defaults are [bambi](https://bambinos.github.io/bambi/)'s weakly informative priors on that
  scale.
- To state a prior in the data's units, divide by the data's SDs, as `slope` and `points` do
  below.

Two priors besides the default:

- **informed**: draws on a pilot study that found about 1.5 points per hour of study.
- **skeptical**: doubts that study time matters.

In [3]:
sd = df[['score', 'hours', 'commute']].std(ddof=0)  # the SDs sample standardizes with


def slope(term, value):
    """A slope of `value` points per unit of `term`, on the standardized scale."""
    return value * sd[term] / sd['score']


def points(value):
    """An SD of `value` points, on the standardized scale."""
    return value / sd['score']


informed = {
    # centered on the pilot study's 1.5 points per hour
    'fixed': {'family': 'student', 'hours': {'nu': slope('hours', 1.5), 'tau': slope('hours', 1)}},
    'random_sd': {  # mean SDs of the school intercepts and slopes
        'family': 'exponential',
        'Intercept': {'tau': points(5)},
        'hours': {'tau': slope('hours', 1)},
    },
    'sigma_eps': {'family': 'halfnormal', 'tau': points(10)},
    'corr_rfx': {'eta': 2},
}
skeptical = {'fixed': {'hours': {'tau': slope('hours', 1)}}}  # N(0, 1 point per hour)
priors = {'default': None, 'informed': informed, 'skeptical': skeptical}  # None: the defaults

## Multiple priors in one call

A dictionary of named priors fits them all in one call, one row per prior.

In [4]:
%%time
torch.manual_seed(0)  # for reproducibility
formula = 'score ~ hours + commute + (1 + hours | school)'
result = mb.sample(df, formula=formula, priors=priors)

CPU times: user 217 ms, sys: 15.2 ms, total: 232 ms
Wall time: 161 ms


### Posterior summaries

`posteriorSummary(result, batch_index=i)` prints row `i`, in the dictionary's order.<br>
The prior lines are in the data's units.

#### default

The intercept's prior is set at the predictors' means but stated at `hours` = `commute` = 0.<br>
On the way there, it picks up the slopes' uncertainty and widens.

In [5]:
print(mb.posteriorSummary(result, batch_index=0))

Formula:  score ~ hours + commute + (1 + hours | school)
n = 137   m = 15
Priors:
  Intercept ~ N(71.2, 97.23)
  hours ~ N(0, 13.26)
  commute ~ N(0, 2.432)
  σ_Intercept ~ HN(23.57) at mean hours
  σ_hours ~ HN(13.26)
  Corr ~ LKJ(1) at mean hours
  σ_Residual ~ HT₅(9.426)

Fixed Effects:
|           |   Mean |    SD |   2.5% |   50.0% |   97.5% |   P(>0) |   Contr. |
|-----------|--------|-------|--------|---------|---------|---------|----------|
| Intercept | 62.917 | 2.968 | 57.283 |  63.044 |  68.888 |   1.000 |    0.999 |
| hours     |  2.105 | 0.392 |  1.344 |   2.098 |   2.900 |   1.000 |    0.999 |
| commute   | -0.064 | 0.067 | -0.196 |  -0.062 |   0.061 |   0.158 |    0.999 |

Standard Deviations:
|           |   Mean |    SD |   2.5% |   50.0% |   97.5% |
|-----------|--------|-------|--------|---------|---------|
| Intercept |  6.439 | 2.586 |  2.829 |   6.212 |  12.572 |
| hours     |  0.564 | 0.477 |  0.016 |   0.434 |   1.604 |
| Residual  |  7.247 | 0.466 |  6.404 |   

#### informed

The prior lines read back the informed priors in points.<br>
A line ending in "at mean …" has no closed form at 0, so it keeps the prior where it was set.

In [6]:
print(mb.posteriorSummary(result, batch_index=1))

Formula:  score ~ hours + commute + (1 + hours | school)
n = 137   m = 15
Priors:
  Intercept ~ t₅(71.2, 23.57) at mean hours, commute
  hours ~ t₅(1.5, 1)
  commute ~ t₅(0, 2.432)
  σ_Intercept ~ Exp(5) at mean hours
  σ_hours ~ Exp(1)
  Corr ~ LKJ(2) at mean hours
  σ_Residual ~ HN(10)

Fixed Effects:
|           |   Mean |    SD |   2.5% |   50.0% |   97.5% |   P(>0) |   Contr. |
|-----------|--------|-------|--------|---------|---------|---------|----------|
| Intercept | 63.215 | 3.146 | 56.183 |  63.375 |  69.054 |   1.000 |    0.999 |
| hours     |  2.009 | 0.370 |  1.315 |   2.005 |   2.682 |   1.000 |    0.918 |
| commute   | -0.056 | 0.066 | -0.185 |  -0.058 |   0.075 |   0.198 |    1.000 |

Standard Deviations:
|           |   Mean |    SD |   2.5% |   50.0% |   97.5% |
|-----------|--------|-------|--------|---------|---------|
| Intercept |  5.987 | 1.636 |  3.232 |   5.808 |   9.920 |
| hours     |  0.381 | 0.320 |  0.013 |   0.312 |   1.209 |
| Residual  |  7.356 | 0.496

#### skeptical

The `hours` prior is narrow enough to compete with the data, so its contraction (Contr.) drops.

In [7]:
print(mb.posteriorSummary(result, batch_index=2))

Formula:  score ~ hours + commute + (1 + hours | school)
n = 137   m = 15
Priors:
  Intercept ~ N(71.2, 73.46)
  hours ~ N(0, 1)
  commute ~ N(0, 2.432)
  σ_Intercept ~ HN(23.57) at mean hours
  σ_hours ~ HN(13.26)
  Corr ~ LKJ(1) at mean hours
  σ_Residual ~ HT₅(9.426)

Fixed Effects:
|           |   Mean |    SD |   2.5% |   50.0% |   97.5% |   P(>0) |   Contr. |
|-----------|--------|-------|--------|---------|---------|---------|----------|
| Intercept | 63.851 | 3.072 | 57.947 |  64.063 |  69.655 |   1.000 |    0.998 |
| hours     |  1.826 | 0.387 |  1.069 |   1.833 |   2.603 |   1.000 |    0.850 |
| commute   | -0.055 | 0.067 | -0.186 |  -0.061 |   0.076 |   0.214 |    0.999 |

Standard Deviations:
|           |   Mean |    SD |   2.5% |   50.0% |   97.5% |
|-----------|--------|-------|--------|---------|---------|
| Intercept |  6.425 | 2.159 |  2.514 |   6.267 |  11.605 |
| hours     |  0.561 | 0.482 |  0.025 |   0.436 |   2.053 |
| Residual  |  7.349 | 0.507 |  6.493 |   7.30

### Side by side

The table lines up each prior's posterior means and 95% intervals with the true values:

- Every interval contains the true value.
- The skeptical prior pulls `hours` toward 0.
- `σ_hours` moves most: 15 schools say little about how the slope varies.

In [8]:
fixed, random = result.param_names['ffx'], result.param_names['sigma_rfx']


def estimates(result, i):
    """Posterior mean [95% interval] of row i's global parameters, in the data's units."""
    si, post = result.scale_info, result.proposal
    ffx = si.to_original_scale(post.ffx[i])[:, : len(fixed)]  # (S, d)
    sigma, _ = si.sdCorrToOriginal(post.sigma_rfx[i], post.corr_rfx[i])
    draws = torch.cat([ffx, sigma[:, : len(random)], post.sigma_eps[i, :, None]], -1).numpy()
    lo, hi = np.percentile(draws, [2.5, 97.5], axis=0)
    return [f'{m:.2f} [{l:.2f}, {h:.2f}]' for m, l, h in zip(draws.mean(0), lo, hi)]


names = fixed + [f'σ_{t}' for t in random] + ['σ_Residual']
side_by_side = pd.DataFrame(
    {name: estimates(result, i) for i, name in enumerate(priors)},
    index=pd.Index(names, name='mean [95% interval]'),
)
side_by_side.insert(0, 'truth', truth[names])
side_by_side

,truth,default,informed,skeptical
mean [95% interval],,,,
Intercept,60.0,"62.92 [57.28, 68.89]","63.21 [56.18, 69.05]","63.85 [57.95, 69.65]"
hours,2.0,"2.11 [1.34, 2.90]","2.01 [1.32, 2.68]","1.83 [1.07, 2.60]"
commute,0.0,"-0.06 [-0.20, 0.06]","-0.06 [-0.18, 0.07]","-0.06 [-0.19, 0.08]"
σ_Intercept,6.0,"6.44 [2.83, 12.57]","5.99 [3.23, 9.92]","6.43 [2.51, 11.61]"
σ_hours,1.0,"0.56 [0.02, 1.60]","0.38 [0.01, 1.21]","0.56 [0.03, 2.05]"
σ_Residual,8.0,"7.25 [6.40, 8.28]","7.36 [6.46, 8.55]","7.35 [6.49, 8.43]"


## Bayes factors

`evidence=True` adds `result.evidence`:

- **`log_evidence`**: the log marginal likelihood log p(y), one per row.<br>
  It measures how probable the data are under the row's model, priors included.
- **`psis_k`**: the estimate's Pareto k̂, from Pareto-smoothed importance sampling (PSIS).<br>
  `sample` warns above 0.7.

### Does `commute` matter?

1. We compare the models with and without it.
2. A Bayes factor depends on the prior of any parameter only one model has.
3. So we fit the model with `commute` under three priors for its slope.

In [9]:
commute_priors = {  # the slope's prior SD, in points per minute
    'default': None,
    'medium': {'fixed': {'commute': {'tau': slope('commute', 0.5)}}},
    'narrow': {'fixed': {'commute': {'tau': slope('commute', 0.1)}}},
}
torch.manual_seed(0)
result_with = mb.sample(df, formula=formula, priors=commute_priors, evidence=True)
torch.manual_seed(0)
result_without = mb.sample(df, formula='score ~ hours + (1 + hours | school)', evidence=True)

The log Bayes factor (BF) is the difference of the two models' `log_evidence`.<br>
Above 0, it favors leaving `commute` out.<br>
[Kass and Raftery (1995)](https://doi.org/10.1080/01621459.1995.10476572) grade its size:

In [10]:
def strength(log_bf):
    """How strongly a Bayes factor favors its model: Kass and Raftery (1995), JASA 90:773."""
    bf = np.exp(abs(log_bf))
    for bound, label in [(3, 'not worth more than a bare mention'), (20, 'positive'), (150, 'strong')]:
        if bf < bound:
            return label
    return 'very strong'


p_with = result_with.prior_params['tau_ffx'][:, fixed.index('commute')]
log_ev_with = result_with.evidence['log_evidence']
log_ev_without = result_without.evidence['log_evidence'][0]
log_bf = log_ev_without - log_ev_with
bayes_factors = pd.DataFrame(
    {
        'SD (points per minute)': p_with / slope('commute', 1),
        'log p(y) with commute': log_ev_with,
        'PSIS k̂': result_with.evidence['psis_k'],
        'log BF': log_bf,
        'BF': np.exp(log_bf),
        'evidence': [strength(b) for b in log_bf],
    },
    index=pd.Index(list(commute_priors), name='commute prior'),
)
print(f'log p(y) without commute: {log_ev_without:.2f}')
bayes_factors.round(2)

log p(y) without commute: -493.34


,SD (points per minute),log p(y) with commute,PSIS k̂,log BF,BF,evidence
commute prior,,,,,,
default,2.43,-496.49,0.22,3.16,23.54,strong
medium,0.50,-494.93,0.41,1.60,4.94,positive
narrow,0.10,-493.60,0.44,0.27,1.31,not worth more than a bare mention


Every BF favors leaving `commute` out, but by how much depends on the slope's prior:

1. The default prior spreads the slope over values the data rule out.
2. The evidence charges the model with `commute` for them.
3. A narrower prior brings the two models' predictions closer, so the BF shrinks.

So report a Bayes factor with the prior it came from.

## Recap

- `priors=` takes `fixed`, `random_sd`, `sigma_eps` and `corr_rfx` on the standardized scale.
- A dictionary of named priors fits them all in one call, one row per prior.
- `evidence=True` adds one log p(y) per row; two rows' difference is a log Bayes factor.